In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TugasMandiri4") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [5]:

df_tugas = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv", header=True, inferSchema=True)


df_tugas.printSchema()


print("Jumlah baris dataset:", df_tugas.count())


df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris dataset: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|

In [7]:
from pyspark.sql.functions import col


jumlah_kosong = df_tugas.filter(col("rating").isNull()).count()
print("Jumlah data kosong pada kolom rating:", jumlah_kosong)


df_bersih = df_tugas.na.fill({"rating": 0})


print("Sisa data kosong setelah dibersihkan:", df_bersih.filter(col("rating").isNull()).count())

# Alasan menggunakan na.fill():
# Saya memilih menggunakan df.na.fill({"rating": 0}) untuk mengisi nilai kosong dengan angka 0 daripada menggunakan df.na.drop(). Jika kita menggunakan drop(), kita akan membuang seluruh baris transaksi tersebut. Hal ini sangat berbahaya untuk dataset e-commerce karena kita akan kehilangan data pendapatan finansial yang sah (transaksi benar-benar terjadi dan dibayar, hanya saja pembeli tidak memberikan rating).

Jumlah data kosong pada kolom rating: 204
Sisa data kosong setelah dibersihkan: 0


In [8]:
from pyspark.sql.functions import when


df_trans = df_bersih.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))


df_trans = df_trans.withColumn("tier_transaksi", 
                               when(col("total_pendapatan") > 500000, "Besar")
                               .otherwise("Kecil"))

df_trans.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [9]:
from pyspark.sql.functions import sum as spark_sum, avg


print("1. Kategori dengan total_pendapatan tertinggi:")
df_trans.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total")) \
    .orderBy(col("total").desc()) \
    .show(1)


print("2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:")
df_trans.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(1)


print("3. Rata-rata rating per metode pembayaran:")
df_trans.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .show()

1. Kategori dengan total_pendapatan tertinggi:


+------------+---------+
|    kategori|    total|
+------------+---------+
|Rumah Tangga|138665000|
+------------+---------+
only showing top 1 row

2. Kota dengan jumlah transaksi tier 'Besar' terbanyak:
+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row

3. Rata-rata rating per metode pembayaran:
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



In [10]:
# Jawaban Bagian E
# Menyimpan ke HDFS dengan format CSV
df_trans.write.csv("hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september", header=True, mode="overwrite")

print("Data berhasil disimpan ke HDFS!")

[Stage 27:>                                                         (0 + 1) / 1]

Data berhasil disimpan ke HDFS!
